<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/05_GES_Aware_Genomic_RAG_Cell_7B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES-RAG Experiment 2 — Cell 7B1


In [3]:
from collections import OrderedDict
from pathlib import Path
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# CELL 7B1 — SCORE-BLIND EVIDENCE / QUESTION PREFLIGHT
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG_Cell_7B1_V3.ipynb"
CELL_ID = "7B1"
PACKAGE_VERSION = "v1"

CONFIG_DIR = ROOT / "configs" / "stage7_rag"
TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"
QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for directory in (CONFIG_DIR, TABLE_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# FROZEN CELL 7B0 INPUTS
# ============================================================

CELL_7B0_PROTOCOL = (
    CONFIG_DIR
    / "cell_7b0_downstream_rag_protocol_v1.json"
)

CELL_7B0_CONDITIONS = (
    TABLE_DIR
    / "cell_7b0_experimental_condition_inventory_v1.csv"
)

CELL_7B0_METRICS = (
    TABLE_DIR
    / "cell_7b0_evaluation_metric_inventory_v1.csv"
)

CELL_7B0_CONTROLS = (
    TABLE_DIR
    / "cell_7b0_leakage_and_invariance_control_inventory_v1.csv"
)

CELL_7B0_QC = (
    QC_DIR
    / "cell_7b0_downstream_rag_protocol_qc_v1.json"
)

CELL_7B0_MANIFEST = (
    CONFIG_DIR
    / "cell_7b0_downstream_rag_protocol_manifest_v1.json"
)


# ============================================================
# FROZEN RAW-T1 ANCESTRY
# ============================================================

CELL_7A1_MANIFEST = (
    CONFIG_DIR
    / "cell_7a1_t1_corpus_source_preflight_manifest_v1.json"
)

T1_PARQUET = (
    ROOT
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

T1_FREEZE_MANIFEST = (
    ROOT
    / "configs"
    / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)


# This path is defined only as a prohibition guard.
# It is never opened, hashed, inspected, or loaded in Cell 7B1.

PROHIBITED_CELL_7A3_SCORE_TABLE = (
    ROOT
    / "data_processed"
    / "stage7_rag"
    / "cell_7a3_t1_frozen_ges_and_metadata_scores_v1.parquet"
)


EXPECTED_HASHES = OrderedDict(
    [
        (
            "cell_7b0_protocol",
            "db4fe2b527e37aba4b4ea5967517c3896e933f989a7ad264406077c4298bd849",
        ),
        (
            "cell_7b0_conditions",
            "ca1b51cbf21e9401da90c9fa0688e71f9d3755cbd467220b9409c2b79425d03c",
        ),
        (
            "cell_7b0_metrics",
            "e4e3a00870cb8e1430f01a3635ee7b6a77b7ceb3bc2ca0fad495e170cf451223",
        ),
        (
            "cell_7b0_controls",
            "bc8e8ec7f1d120bc57589ae544d87b0ab7f8fcc26755cf89c34e782787bcf049",
        ),
        (
            "cell_7b0_qc",
            "126207c1552ba4a65eb4a60e36aef1e95500228b4bfc38b77f8d773dd3a3d776",
        ),
        (
            "cell_7b0_manifest",
            "df9342b8a2fb641f4cb68ff18ae9568bb7f63eff3fcc5e1ae1106601fa59e42a",
        ),
        (
            "cell_7a1_manifest",
            "84e509d97f01fb8dc0b6ad0c7e24de762e8923b9068fc835bf328105f79e946d",
        ),
        (
            "t1_parquet",
            "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c",
        ),
        (
            "t1_freeze_manifest",
            "7eaeff0fee3df96973130f721d6c1f2a02fd9108e85af7b2743cdd7750a4372e",
        ),
    ]
)


INPUT_PATHS = OrderedDict(
    [
        ("cell_7b0_protocol", CELL_7B0_PROTOCOL),
        ("cell_7b0_conditions", CELL_7B0_CONDITIONS),
        ("cell_7b0_metrics", CELL_7B0_METRICS),
        ("cell_7b0_controls", CELL_7B0_CONTROLS),
        ("cell_7b0_qc", CELL_7B0_QC),
        ("cell_7b0_manifest", CELL_7B0_MANIFEST),
        ("cell_7a1_manifest", CELL_7A1_MANIFEST),
        ("t1_parquet", T1_PARQUET),
        ("t1_freeze_manifest", T1_FREEZE_MANIFEST),
    ]
)


EXPECTED_CELL_7B0_DECISION = (
    "PASS_STAGE7B0_DOWNSTREAM_RAG_PROTOCOL_FROZEN_CHECKSUM_PROTECTED_"
    "STAGE7A3_REVERIFIED_PRIMARY_SOFT_RERANKING_AND_MANDATORY_COMPARATOR_"
    "ARMS_PRESPECIFIED_NO_EVIDENCE_PACKETS_CORPUS_EMBEDDINGS_RETRIEVAL_"
    "QUESTIONS_PROMPTS_OR_LLM_CELL7B1_SCORE_BLIND_EVIDENCE_AND_QUESTION_"
    "PREFLIGHT_ONLY_AUTHORIZED"
)


EXPECTED_CELL_7A1_DECISION = (
    "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
    "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
    "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
)


EXPECTED_ROWS = 100_920
EXPECTED_COLUMNS = 36
EXPECTED_NESTED_SCVS = 145_400
EXPECTED_CONFLICT_POSITIVE = 6_602
EXPECTED_EMPTY_CONDITION_IDS = 730


EXPECTED_GENE_COUNTS = OrderedDict(
    [
        ("BRCA1", 32_603),
        ("BRCA2", 49_221),
        ("MLH1", 13_684),
        ("EGFR", 5_412),
    ]
)


EXPECTED_AXIS_COUNTS = OrderedDict(
    [
        ("GermlineClassification", 97_526),
        ("OncogenicityClassification", 52),
        ("SomaticClinicalImpact", 25),
        ("NoClassification", 3_317),
    ]
)


# ============================================================
# OUTPUTS
# ============================================================

OUTPUTS = OrderedDict(
    [
        (
            "source_inventory",
            TABLE_DIR
            / "cell_7b1_score_blind_source_inventory_v1.csv",
        ),
        (
            "field_derivability",
            TABLE_DIR
            / "cell_7b1_evidence_field_derivability_inventory_v1.csv",
        ),
        (
            "gene_axis_inventory",
            TABLE_DIR
            / "cell_7b1_gene_axis_eligibility_inventory_v1.csv",
        ),
        (
            "question_strata",
            TABLE_DIR
            / "cell_7b1_question_stratum_availability_inventory_v1.csv",
        ),
        (
            "preflight_report",
            QC_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_report_v1.json",
        ),
        (
            "qc",
            QC_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_qc_v1.json",
        ),
        (
            "manifest",
            CONFIG_DIR
            / "cell_7b1_score_blind_evidence_question_preflight_manifest_v1.json",
        ),
    ]
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def sidecar_path(path):
    return Path(str(path) + ".sha256")


def read_sidecar_hash(path):
    text = Path(path).read_text(
        encoding="utf-8"
    )

    values = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not values:
        raise ValueError(
            f"No SHA-256 found in sidecar: {path}"
        )

    return values[0].lower()


def sidecar_is_valid(path):
    path = Path(path)
    checksum_sidecar = sidecar_path(path)

    return (
        path.exists()
        and checksum_sidecar.exists()
        and read_sidecar_hash(checksum_sidecar)
        == sha256_file(path)
    )


def verify_exact_hash(
    label,
    path,
    expected,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing frozen artifact for {label}: {path}"
        )

    observed = sha256_file(path)

    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}\n"
            f"Expected: {expected}\n"
            f"Observed: {observed}\n"
            f"Path: {path}"
        )

    return observed


def json_native(value):
    if isinstance(value, dict):
        return {
            str(key): json_native(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            json_native(item)
            for item in value
        ]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None

        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if value is pd.NA:
        return None

    if (
        isinstance(value, float)
        and not np.isfinite(value)
    ):
        return None

    return value


def stable_write_bytes(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp-"
        f"{os.getpid()}-"
        f"{time.time_ns()}"
    )

    temporary_path.write_bytes(payload)

    proposed_hash = sha256_file(
        temporary_path
    )

    if path.exists():
        existing_hash = sha256_file(path)

        if existing_hash != proposed_hash:
            temporary_path.unlink(
                missing_ok=True
            )

            raise RuntimeError(
                "Refusing to overwrite a nonidentical "
                "frozen Cell 7B1 artifact.\n"
                f"Path: {path}\n"
                f"Existing: {existing_hash}\n"
                f"Proposed: {proposed_hash}"
            )

        temporary_path.unlink(
            missing_ok=True
        )

    else:
        os.replace(
            temporary_path,
            path,
        )

    return sha256_file(path)


def stable_write_json(
    path,
    payload,
):
    encoded = (
        json.dumps(
            json_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        encoded,
    )


def stable_write_csv(
    path,
    dataframe,
):
    encoded = dataframe.to_csv(
        index=False,
        lineterminator="\n",
    ).encode("utf-8")

    return stable_write_bytes(
        path,
        encoded,
    )


def write_sidecar(path):
    payload = (
        f"{sha256_file(path)}  "
        f"{Path(path).name}\n"
    ).encode("utf-8")

    return stable_write_bytes(
        sidecar_path(path),
        payload,
    )


def normalize_qc_count(
    value,
    field_name,
):
    if isinstance(value, bool):
        return int(value)

    if isinstance(
        value,
        (int, float),
    ):
        return int(value)

    if isinstance(
        value,
        (list, tuple, set, dict),
    ):
        return len(value)

    if (
        isinstance(value, str)
        and value.strip().isdigit()
    ):
        return int(value.strip())

    if value is None:
        return 0

    raise TypeError(
        f"Unsupported QC field type for "
        f"{field_name}: "
        f"{type(value).__name__}"
    )


def qc_summary_counts(payload):
    passed = normalize_qc_count(
        payload.get(
            "passed_checks",
            0,
        ),
        "passed_checks",
    )

    failed = normalize_qc_count(
        payload.get(
            "failed_checks",
            0,
        ),
        "failed_checks",
    )

    total_raw = payload.get(
        "total_checks"
    )

    if total_raw is None:
        checks = payload.get(
            "checks"
        )

        if isinstance(
            checks,
            (list, tuple, dict),
        ):
            total = len(checks)

        else:
            total = passed + failed

    else:
        total = normalize_qc_count(
            total_raw,
            "total_checks",
        )

    return (
        passed,
        failed,
        total,
    )


def normalize_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


def resolve_column(
    columns,
    aliases,
    required=False,
    token_fallback=None,
):
    normalized = {
        normalize_column_name(column): str(column)
        for column in columns
    }

    for alias in aliases:
        normalized_alias = normalize_column_name(
            alias
        )

        if normalized_alias in normalized:
            return normalized[
                normalized_alias
            ]

    if token_fallback:
        matches = [
            str(column)
            for column in columns
            if all(
                token
                in normalize_column_name(column)
                for token in token_fallback
            )
        ]

        if len(matches) == 1:
            return matches[0]

    if required:
        raise KeyError(
            "Could not resolve required column. "
            f"Aliases={aliases}; "
            f"tokens={token_fallback}; "
            f"available={list(columns)}"
        )

    return None


def parse_json_value(value):
    if (
        value is None
        or value is pd.NA
    ):
        return (
            None,
            "missing",
        )

    try:
        if pd.isna(value):
            return (
                None,
                "missing",
            )

    except Exception:
        pass

    if isinstance(
        value,
        (dict, list),
    ):
        return (
            value,
            "native",
        )

    text = str(value).strip()

    if not text:
        return (
            None,
            "blank",
        )

    try:
        return (
            json.loads(text),
            "parsed",
        )

    except json.JSONDecodeError:
        return (
            None,
            "parse_error",
        )


def normalize_gene(value):
    parsed, status = parse_json_value(
        value
    )

    if status == "parse_error":
        parsed = [
            str(value).strip()
        ]

    if isinstance(parsed, str):
        parsed = [parsed]

    if not isinstance(parsed, list):
        return ""

    allowed = set(
        EXPECTED_GENE_COUNTS
    )

    genes = sorted(
        {
            str(item).strip().upper()
            for item in parsed
            if str(item).strip().upper()
            in allowed
        }
    )

    if len(genes) == 1:
        return genes[0]

    return ""


def boolish(value):
    if (
        value is None
        or value is pd.NA
    ):
        return np.nan

    try:
        if pd.isna(value):
            return np.nan

    except Exception:
        pass

    if isinstance(
        value,
        (bool, np.bool_),
    ):
        return bool(value)

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):
        if float(value) == 1.0:
            return True

        if float(value) == 0.0:
            return False

        return np.nan

    text = str(value).strip().lower()

    if text in {
        "true",
        "t",
        "yes",
        "y",
        "1",
    }:
        return True

    if text in {
        "false",
        "f",
        "no",
        "n",
        "0",
    }:
        return False

    return np.nan


def value_is_present(value):
    if (
        value is None
        or value is pd.NA
    ):
        return False

    try:
        if pd.isna(value):
            return False

    except Exception:
        pass

    if isinstance(value, str):
        return bool(
            value.strip()
        )

    if isinstance(
        value,
        (
            list,
            dict,
            tuple,
            set,
        ),
    ):
        return len(value) > 0

    return True


def json_list_length(value):
    parsed, status = parse_json_value(
        value
    )

    if status == "parse_error":
        return (
            np.nan,
            status,
        )

    if parsed is None:
        return (
            0,
            status,
        )

    if isinstance(parsed, list):
        return (
            len(parsed),
            status,
        )

    return (
        np.nan,
        "not_list",
    )


def json_dict_positive_key_count(value):
    parsed, status = parse_json_value(
        value
    )

    if status == "parse_error":
        return (
            np.nan,
            status,
        )

    if parsed is None:
        return (
            0,
            status,
        )

    if not isinstance(parsed, dict):
        return (
            np.nan,
            "not_dict",
        )

    positive = 0

    for item in parsed.values():
        try:
            number = float(item)

        except (
            TypeError,
            ValueError,
        ):
            return (
                np.nan,
                "invalid_numeric",
            )

        if (
            not np.isfinite(number)
            or number < 0
        ):
            return (
                np.nan,
                "invalid_numeric",
            )

        positive += int(
            number > 0
        )

    return (
        positive,
        status,
    )


# ============================================================
# VERIFY CELL 7B0 AND RAW-T1 AUTHORIZATION
# ============================================================

observed_hashes = OrderedDict()

for key, path in INPUT_PATHS.items():
    observed_hashes[key] = verify_exact_hash(
        key,
        path,
        EXPECTED_HASHES[key],
    )

    if not sidecar_is_valid(path):
        raise AssertionError(
            "Missing or invalid SHA-256 sidecar "
            f"for {key}: {path}"
        )


cell_7b0_manifest = json.loads(
    CELL_7B0_MANIFEST.read_text(
        encoding="utf-8"
    )
)

cell_7b0_qc = json.loads(
    CELL_7B0_QC.read_text(
        encoding="utf-8"
    )
)

cell_7a1_manifest = json.loads(
    CELL_7A1_MANIFEST.read_text(
        encoding="utf-8"
    )
)


(
    manifest_qc_passed,
    manifest_qc_failed,
    manifest_qc_total,
) = qc_summary_counts(
    cell_7b0_manifest.get(
        "qc",
        {},
    )
)


(
    record_qc_passed,
    record_qc_failed,
    record_qc_total,
) = qc_summary_counts(
    cell_7b0_qc
)


if (
    cell_7b0_manifest.get(
        "terminal_decision"
    )
    != EXPECTED_CELL_7B0_DECISION
):
    raise AssertionError(
        "Cell 7B0 terminal decision is not "
        "the expected frozen PASS."
    )


next_7b0 = cell_7b0_manifest.get(
    "next_authorized_cell",
    {},
)


if (
    next_7b0.get("cell_id")
    != "7B1"
):
    raise AssertionError(
        "Cell 7B0 does not authorize Cell 7B1."
    )


if (
    next_7b0.get(
        "may_load_cell_7a3_scores"
    )
    is not False
):
    raise AssertionError(
        "Cell 7B0 score-blind restriction "
        "is not preserved."
    )


if (
    cell_7a1_manifest.get(
        "terminal_decision"
    )
    != EXPECTED_CELL_7A1_DECISION
):
    raise AssertionError(
        "Cell 7A1 source-preflight ancestry "
        "is not the expected PASS."
    )


immutable_hashes_before = {
    key: sha256_file(path)
    for key, path in INPUT_PATHS.items()
}


# ============================================================
# LOAD ONLY RAW T1
# ============================================================

parquet_metadata = pq.ParquetFile(
    T1_PARQUET
).metadata


if (
    int(parquet_metadata.num_rows)
    != EXPECTED_ROWS
):
    raise AssertionError(
        f"T1 rows={parquet_metadata.num_rows}; "
        f"expected {EXPECTED_ROWS}."
    )


if (
    int(parquet_metadata.num_columns)
    != EXPECTED_COLUMNS
):
    raise AssertionError(
        f"T1 columns={parquet_metadata.num_columns}; "
        f"expected {EXPECTED_COLUMNS}."
    )


t1 = pd.read_parquet(
    T1_PARQUET
).copy()


resolved = OrderedDict(
    [
        (
            "rcv_accession",
            resolve_column(
                t1.columns,
                [
                    "rcv_accession",
                    "t1_rcv_accession",
                ],
                required=True,
                token_fallback=[
                    "rcv",
                    "accession",
                ],
            ),
        ),
        (
            "vcv_accession",
            resolve_column(
                t1.columns,
                [
                    "vcv_accession",
                    "variation_archive_accession",
                ],
                token_fallback=[
                    "vcv",
                    "accession",
                ],
            ),
        ),
        (
            "variation_id",
            resolve_column(
                t1.columns,
                [
                    "variation_id",
                    "variationid",
                ],
                token_fallback=[
                    "variation",
                    "id",
                ],
            ),
        ),
        (
            "variation_name",
            resolve_column(
                t1.columns,
                [
                    "variation_name",
                    "variant_name",
                    "name",
                ],
                token_fallback=[
                    "variation",
                    "name",
                ],
            ),
        ),
        (
            "target_gene",
            resolve_column(
                t1.columns,
                [
                    "target_genes_json",
                    "target_gene",
                    "gene",
                ],
                required=True,
            ),
        ),
        (
            "classification_axis",
            resolve_column(
                t1.columns,
                [
                    "aggregate_classification_axis",
                    "classification_axis",
                ],
                required=True,
            ),
        ),
        (
            "condition_names",
            resolve_column(
                t1.columns,
                [
                    "condition_names_json",
                    "condition_names",
                    "trait_names_json",
                ],
                token_fallback=[
                    "condition",
                    "name",
                ],
            ),
        ),
        (
            "condition_ids",
            resolve_column(
                t1.columns,
                [
                    "condition_ids_json",
                    "condition_identifiers_json",
                    "condition_xrefs_json",
                    "trait_ids_json",
                ],
                token_fallback=[
                    "condition",
                    "id",
                ],
            ),
        ),
        (
            "aggregate_classification",
            resolve_column(
                t1.columns,
                [
                    "aggregate_classification",
                    "aggregate_clinical_significance",
                    "aggregate_classification_description",
                    "clinical_significance",
                ],
                token_fallback=[
                    "aggregate",
                    "classification",
                ],
            ),
        ),
        (
            "aggregate_classification_group",
            resolve_column(
                t1.columns,
                [
                    "aggregate_classification_group",
                    "classification_group",
                    "aggregate_group",
                ],
                token_fallback=[
                    "classification",
                    "group",
                ],
            ),
        ),
        (
            "aggregate_review_status",
            resolve_column(
                t1.columns,
                [
                    "aggregate_review_status",
                    "review_status",
                ],
                token_fallback=[
                    "review",
                    "status",
                ],
            ),
        ),
        (
            "aggregate_review_stars",
            resolve_column(
                t1.columns,
                [
                    "aggregate_review_stars",
                    "review_stars",
                ],
                required=True,
                token_fallback=[
                    "review",
                    "star",
                ],
            ),
        ),
        (
            "aggregate_conflict_flag",
            resolve_column(
                t1.columns,
                [
                    "aggregate_conflict_flag",
                    "conflict_flag",
                ],
                required=True,
                token_fallback=[
                    "conflict",
                    "flag",
                ],
            ),
        ),
        (
            "aggregate_last_evaluated",
            resolve_column(
                t1.columns,
                [
                    "aggregate_last_evaluated",
                    "last_evaluated",
                ],
                required=True,
                token_fallback=[
                    "last",
                    "evaluated",
                ],
            ),
        ),
        (
            "scv_count",
            resolve_column(
                t1.columns,
                [
                    "scv_count_xml",
                    "scv_count",
                ],
                required=True,
                token_fallback=[
                    "scv",
                    "count",
                ],
            ),
        ),
        (
            "unique_submitter_count",
            resolve_column(
                t1.columns,
                [
                    "unique_submitter_count_xml",
                    "unique_submitter_count",
                ],
                required=True,
                token_fallback=[
                    "submitter",
                    "count",
                ],
            ),
        ),
        (
            "submitter_ids",
            resolve_column(
                t1.columns,
                [
                    "submitter_ids_json",
                    "submitter_org_ids_json",
                    "unique_submitter_ids_json",
                ],
                token_fallback=[
                    "submitter",
                    "id",
                ],
            ),
        ),
        (
            "scv_group_counts",
            resolve_column(
                t1.columns,
                [
                    "scv_group_counts_json",
                    "group_counts_json",
                ],
                required=True,
                token_fallback=[
                    "scv",
                    "group",
                    "count",
                ],
            ),
        ),
        (
            "nested_scvs",
            resolve_column(
                t1.columns,
                [
                    "scv_records_json",
                    "nested_scvs_json",
                    "nested_scv_assertions_json",
                    "scvs_json",
                    "nested_scvs",
                ],
                required=True,
                token_fallback=[
                    "scv",
                    "records",
                ],
            ),
        ),
        (
            "embedded_cutoff_date",
            resolve_column(
                t1.columns,
                [
                    "embedded_data_cutoff_date",
                    "data_cutoff_date",
                    "embedded_cutoff_date",
                ],
                required=True,
                token_fallback=[
                    "cutoff",
                    "date",
                ],
            ),
        ),
        (
            "release_label",
            resolve_column(
                t1.columns,
                [
                    "archive_release_label",
                    "release_label",
                    "clinvar_release_label",
                ],
                token_fallback=[
                    "release",
                    "label",
                ],
            ),
        ),
        (
            "source_filename",
            resolve_column(
                t1.columns,
                [
                    "source_filename",
                    "source_file_name",
                    "xml_source_filename",
                ],
                token_fallback=[
                    "source",
                    "filename",
                ],
            ),
        ),
        (
            "source_sha256",
            resolve_column(
                t1.columns,
                [
                    "source_sha256",
                    "source_file_sha256",
                    "xml_source_sha256",
                ],
                token_fallback=[
                    "source",
                    "sha256",
                ],
            ),
        ),
    ]
)


if (
    resolved["aggregate_classification"]
    is None
    and resolved[
        "aggregate_classification_group"
    ]
    is None
):
    raise KeyError(
        "Aggregate classification fields "
        "could not be resolved."
    )


if (
    resolved["condition_names"]
    is None
    and resolved["condition_ids"]
    is None
):
    raise KeyError(
        "Condition fields could not be resolved."
    )


# ============================================================
# SCORE-BLIND DERIVATIONS
# ============================================================

rcv = (
    t1[
        resolved["rcv_accession"]
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)


gene = t1[
    resolved["target_gene"]
].map(normalize_gene)


axis = (
    t1[
        resolved["classification_axis"]
    ]
    .astype("string")
    .fillna("NoClassification")
    .str.strip()
    .replace(
        "",
        "NoClassification",
    )
)


conflict = t1[
    resolved["aggregate_conflict_flag"]
].map(boolish)


review_stars = pd.to_numeric(
    t1[
        resolved["aggregate_review_stars"]
    ],
    errors="coerce",
)


last_evaluated = pd.to_datetime(
    t1[
        resolved[
            "aggregate_last_evaluated"
        ]
    ],
    errors="coerce",
    utc=True,
)


scv_count = pd.to_numeric(
    t1[
        resolved["scv_count"]
    ],
    errors="coerce",
)


submitter_count = pd.to_numeric(
    t1[
        resolved["unique_submitter_count"]
    ],
    errors="coerce",
)


nested_results = t1[
    resolved["nested_scvs"]
].map(json_list_length)


nested_scv_count = pd.Series(
    [
        item[0]
        for item in nested_results
    ],
    index=t1.index,
    dtype=float,
)


nested_status = pd.Series(
    [
        item[1]
        for item in nested_results
    ],
    index=t1.index,
    dtype="string",
)


group_results = t1[
    resolved["scv_group_counts"]
].map(json_dict_positive_key_count)


positive_group_count = pd.Series(
    [
        item[0]
        for item in group_results
    ],
    index=t1.index,
    dtype=float,
)


group_status = pd.Series(
    [
        item[1]
        for item in group_results
    ],
    index=t1.index,
    dtype="string",
)


if resolved["condition_ids"] is not None:
    condition_id_results = t1[
        resolved["condition_ids"]
    ].map(json_list_length)

    condition_id_count = pd.Series(
        [
            item[0]
            for item in condition_id_results
        ],
        index=t1.index,
        dtype=float,
    )

else:
    condition_id_count = pd.Series(
        np.nan,
        index=t1.index,
        dtype=float,
    )


if resolved["condition_names"] is not None:
    condition_name_present = t1[
        resolved["condition_names"]
    ].map(value_is_present)

else:
    condition_name_present = pd.Series(
        False,
        index=t1.index,
    )


if (
    resolved["aggregate_classification"]
    is not None
):
    classification_present = t1[
        resolved["aggregate_classification"]
    ].map(value_is_present)

else:
    classification_present = pd.Series(
        False,
        index=t1.index,
    )


if (
    resolved[
        "aggregate_classification_group"
    ]
    is not None
):
    classification_group_present = t1[
        resolved[
            "aggregate_classification_group"
        ]
    ].map(value_is_present)

else:
    classification_group_present = pd.Series(
        False,
        index=t1.index,
    )


condition_present = (
    condition_name_present
    | condition_id_count
    .fillna(0)
    .gt(0)
)


interpretation_present = (
    classification_present
    | classification_group_present
)


nested_available = (
    nested_scv_count
    .fillna(0)
    .gt(0)
)


summary_eligible = (
    rcv.notna()
    & rcv.ne("")
    & gene.ne("")
    & condition_present
    & interpretation_present
    & nested_available
)


conflict_eligible = (
    summary_eligible
    & (
        conflict
        .fillna(False)
        .astype(bool)
        | positive_group_count
        .fillna(0)
        .gt(1)
    )
)


rigor_eligible = (
    summary_eligible
    & review_stars.notna()
    & review_stars.between(
        0,
        4,
        inclusive="both",
    )
    & submitter_count
    .fillna(0)
    .ge(1)
    & scv_count
    .fillna(0)
    .ge(1)
)


uncertainty_eligible = (
    summary_eligible
    & (
        axis.eq(
            "NoClassification"
        )
        | conflict
        .fillna(False)
        .astype(bool)
        | condition_id_count
        .fillna(0)
        .eq(0)
        | last_evaluated.isna()
        | review_stars
        .fillna(0)
        .le(1)
        | positive_group_count
        .fillna(0)
        .gt(1)
    )
)


eligibility_masks = OrderedDict(
    [
        (
            "aggregate_interpretation_summary",
            summary_eligible,
        ),
        (
            "conflict_recognition",
            conflict_eligible,
        ),
        (
            "evidence_rigor_and_provenance",
            rigor_eligible,
        ),
        (
            "uncertainty_or_abstention",
            uncertainty_eligible,
        ),
    ]
)


# ============================================================
# SOURCE INVENTORY
# ============================================================

source_inventory_rows = []

for key, path in INPUT_PATHS.items():
    record = {
        "artifact_key": key,
        "path": str(path),
        "sha256": observed_hashes[key],
        "sidecar_verified": sidecar_is_valid(
            path
        ),
        "bytes": int(
            path.stat().st_size
        ),
        "rows": None,
        "columns": None,
        "loaded_in_cell_7b1": (
            key == "t1_parquet"
        ),
    }

    if path.suffix.lower() == ".parquet":
        metadata = pq.ParquetFile(
            path
        ).metadata

        record["rows"] = int(
            metadata.num_rows
        )

        record["columns"] = int(
            metadata.num_columns
        )

    source_inventory_rows.append(
        record
    )


source_inventory = pd.DataFrame(
    source_inventory_rows
)


# ============================================================
# FIELD DERIVABILITY INVENTORY
# ============================================================

field_specs = [
    (
        "rcv_accession",
        "identifier",
        "top_level",
        True,
        "direct",
    ),
    (
        "vcv_accession",
        "identifier",
        "top_level",
        False,
        "direct",
    ),
    (
        "variation_id",
        "identifier",
        "top_level",
        False,
        "direct",
    ),
    (
        "variation_name",
        "identifier",
        "top_level",
        False,
        "direct",
    ),
    (
        "target_gene",
        "identifier",
        "top_level",
        True,
        "direct",
    ),
    (
        "release_label",
        "provenance",
        "top_level_or_manifest",
        False,
        "direct_or_manifest",
    ),
    (
        "embedded_cutoff_date",
        "provenance",
        "top_level",
        True,
        "direct",
    ),
    (
        "source_filename",
        "provenance",
        "top_level_or_manifest",
        False,
        "direct_or_manifest",
    ),
    (
        "source_sha256",
        "provenance",
        "top_level_or_manifest",
        False,
        "direct_or_manifest",
    ),
    (
        "condition_names",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "condition_ids",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "aggregate_classification",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "aggregate_classification_group",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "classification_axis",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "aggregate_review_status",
        "clinical_metadata",
        "top_level",
        False,
        "direct",
    ),
    (
        "aggregate_review_stars",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "aggregate_conflict_flag",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "aggregate_last_evaluated",
        "clinical_metadata",
        "top_level",
        True,
        "direct",
    ),
    (
        "scv_count",
        "provenance",
        "top_level",
        True,
        "direct",
    ),
    (
        "unique_submitter_count",
        "provenance",
        "top_level",
        True,
        "direct",
    ),
    (
        "submitter_ids",
        "provenance",
        "top_level_or_nested",
        False,
        "direct_or_nested",
    ),
    (
        "scv_group_counts",
        "nested_summary",
        "top_level",
        True,
        "direct",
    ),
    (
        "nested_scv_assertions",
        "nested_evidence",
        "nested_scvs",
        True,
        "deterministically_parseable",
    ),
    (
        "nested_scv_review_statuses",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "nested_scv_classifications",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "nested_scv_last_evaluated",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "nested_scv_submitter_ids",
        "nested_evidence",
        "nested_scvs",
        False,
        "deterministically_parseable",
    ),
    (
        "evidence_packet_id",
        "derived_identifier",
        "later_formatting_policy",
        False,
        "derivable_later",
    ),
    (
        "semantic_evidence_text",
        "derived_text",
        "later_formatting_policy",
        False,
        "derivable_later",
    ),
    (
        "full_ges_p_stable_t1",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
    (
        "full_ges_instability_risk_t1",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
    (
        "no_star_ges_p_stable_t1",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
    (
        "combined_metadata_instability_risk",
        "score",
        "cell_7a3",
        False,
        "available_but_prohibited_in_7b1",
    ),
]


field_rows = []


for (
    field_name,
    category,
    source_class,
    required,
    derivation_status,
) in field_specs:

    resolved_column = resolved.get(
        field_name
    )

    if source_class == "nested_scvs":
        resolved_column = resolved[
            "nested_scvs"
        ]

        available = (
            resolved_column is not None
        )

        missing_rows = int(
            nested_scv_count
            .fillna(0)
            .eq(0)
            .sum()
        )

    elif source_class == "later_formatting_policy":
        available = True
        missing_rows = None

    elif source_class == "cell_7a3":
        available = True
        missing_rows = None
        resolved_column = None

    elif source_class == "top_level_or_manifest":
        available = (
            resolved_column is not None
            or T1_FREEZE_MANIFEST.exists()
        )

        if resolved_column is not None:
            missing_rows = int(
                (
                    ~t1[
                        resolved_column
                    ].map(
                        value_is_present
                    )
                ).sum()
            )

        else:
            missing_rows = None

    elif source_class == "top_level_or_nested":
        available = (
            resolved_column is not None
            or resolved["nested_scvs"]
            is not None
        )

        if resolved_column is not None:
            missing_rows = int(
                (
                    ~t1[
                        resolved_column
                    ].map(
                        value_is_present
                    )
                ).sum()
            )

        else:
            missing_rows = None

    else:
        available = (
            resolved_column is not None
        )

        if resolved_column is not None:
            missing_rows = int(
                (
                    ~t1[
                        resolved_column
                    ].map(
                        value_is_present
                    )
                ).sum()
            )

        else:
            missing_rows = None

    field_rows.append(
        {
            "field_name": field_name,
            "category": category,
            "source_class": source_class,
            "required_for_future_packet": bool(
                required
            ),
            "derivation_status": derivation_status,
            "resolved_source_column": resolved_column,
            "available_or_derivable": bool(
                available
            ),
            "missing_rows_if_directly_auditable": missing_rows,
            "loaded_or_materialized_in_cell_7b1": (
                source_class
                != "cell_7a3"
            ),
        }
    )


field_derivability = pd.DataFrame(
    field_rows
)


# ============================================================
# GENE × CLASSIFICATION-AXIS INVENTORY
# ============================================================

gene_axis_rows = []


for gene_name in EXPECTED_GENE_COUNTS:
    for axis_name in EXPECTED_AXIS_COUNTS:
        mask = (
            gene.eq(gene_name)
            & axis.eq(axis_name)
        )

        gene_axis_rows.append(
            {
                "target_gene": gene_name,
                "classification_axis": axis_name,
                "row_count": int(
                    mask.sum()
                ),
                "conflict_positive": int(
                    (
                        mask
                        & conflict
                        .fillna(False)
                        .astype(bool)
                    ).sum()
                ),
                "empty_structured_condition_ids": int(
                    (
                        mask
                        & condition_id_count
                        .fillna(0)
                        .eq(0)
                    ).sum()
                ),
                "nested_scv_count": int(
                    nested_scv_count.loc[
                        mask
                    ]
                    .fillna(0)
                    .sum()
                ),
            }
        )


gene_axis_inventory = pd.DataFrame(
    gene_axis_rows
)


# ============================================================
# QUESTION-STRATUM AVAILABILITY
# ============================================================

question_rows = []


for gene_name in EXPECTED_GENE_COUNTS:
    gene_mask = gene.eq(
        gene_name
    )

    for (
        question_type,
        eligibility_mask,
    ) in eligibility_masks.items():

        combined_mask = (
            gene_mask
            & eligibility_mask
        )

        axis_breakdown = {
            axis_name: int(
                (
                    combined_mask
                    & axis.eq(axis_name)
                ).sum()
            )
            for axis_name
            in EXPECTED_AXIS_COUNTS
        }

        question_rows.append(
            {
                "target_gene": gene_name,
                "question_type": question_type,
                "target_question_count": 5,
                "eligible_rcv_count": int(
                    combined_mask.sum()
                ),
                "feasible_for_target": bool(
                    combined_mask.sum()
                    >= 5
                ),
                "classification_axis_breakdown_json": json.dumps(
                    axis_breakdown,
                    sort_keys=True,
                    separators=(
                        ",",
                        ":",
                    ),
                ),
                "questions_selected_or_generated": False,
            }
        )


question_strata = pd.DataFrame(
    question_rows
)


# ============================================================
# FAIL-BEFORE-OUTPUT QC
# ============================================================

observed_gene_counts = (
    gene.value_counts(
        dropna=False
    ).to_dict()
)


observed_axis_counts = (
    axis.value_counts(
        dropna=False
    ).to_dict()
)


malformed_rcv = int(
    (
        ~rcv.str.fullmatch(
            r"RCV\d+(?:\.\d+)?",
            na=False,
        )
    ).sum()
)


duplicate_rcv = int(
    rcv.duplicated(
        keep=False
    ).sum()
)


nested_parse_errors = int(
    nested_status.eq(
        "parse_error"
    ).sum()
)


nested_non_lists = int(
    nested_status.eq(
        "not_list"
    ).sum()
)


group_parse_errors = int(
    group_status.eq(
        "parse_error"
    ).sum()
)


group_invalid = int(
    group_status.isin(
        [
            "not_dict",
            "invalid_numeric",
        ]
    ).sum()
)


nested_total = int(
    nested_scv_count
    .fillna(0)
    .sum()
)


scv_count_mismatches = int(
    (
        ~np.isclose(
            nested_scv_count.to_numpy(
                dtype=float
            ),
            scv_count.to_numpy(
                dtype=float
            ),
            rtol=0.0,
            atol=0.0,
            equal_nan=False,
        )
    ).sum()
)


conflict_positive = int(
    conflict
    .fillna(False)
    .astype(bool)
    .sum()
)


empty_condition_ids = int(
    condition_id_count
    .fillna(0)
    .eq(0)
    .sum()
)


all_required_fields_available = bool(
    field_derivability.loc[
        field_derivability[
            "required_for_future_packet"
        ],
        "available_or_derivable",
    ].all()
)


all_strata_feasible = bool(
    question_strata[
        "feasible_for_target"
    ].all()
)


prewrite_checks = OrderedDict(
    [
        (
            "cell_7b0_protocol_exact_hash",
            observed_hashes[
                "cell_7b0_protocol"
            ]
            == EXPECTED_HASHES[
                "cell_7b0_protocol"
            ],
        ),
        (
            "cell_7b0_manifest_exact_hash",
            observed_hashes[
                "cell_7b0_manifest"
            ]
            == EXPECTED_HASHES[
                "cell_7b0_manifest"
            ],
        ),
        (
            "cell_7b0_terminal_decision_exact",
            cell_7b0_manifest.get(
                "terminal_decision"
            )
            == EXPECTED_CELL_7B0_DECISION,
        ),
        (
            "cell_7b0_authorizes_7b1",
            next_7b0.get(
                "cell_id"
            )
            == "7B1",
        ),
        (
            "cell_7b0_manifest_qc_50_of_50",
            manifest_qc_passed == 50
            and manifest_qc_failed == 0
            and manifest_qc_total == 50,
        ),
        (
            "cell_7b0_record_qc_50_of_50",
            record_qc_passed == 50
            and record_qc_failed == 0
            and record_qc_total == 50,
        ),
        (
            "cell_7a1_manifest_exact_hash",
            observed_hashes[
                "cell_7a1_manifest"
            ]
            == EXPECTED_HASHES[
                "cell_7a1_manifest"
            ],
        ),
        (
            "cell_7a1_terminal_decision_exact",
            cell_7a1_manifest.get(
                "terminal_decision"
            )
            == EXPECTED_CELL_7A1_DECISION,
        ),
        (
            "all_nine_inputs_verified",
            len(observed_hashes)
            == 9,
        ),
        (
            "all_nine_input_sidecars_valid",
            all(
                sidecar_is_valid(path)
                for path
                in INPUT_PATHS.values()
            ),
        ),
        (
            "t1_exact_hash",
            observed_hashes[
                "t1_parquet"
            ]
            == EXPECTED_HASHES[
                "t1_parquet"
            ],
        ),
        (
            "t1_rows_100920",
            len(t1)
            == EXPECTED_ROWS,
        ),
        (
            "t1_columns_36",
            len(t1.columns)
            == EXPECTED_COLUMNS,
        ),
        (
            "rcv_unique_100920",
            rcv.nunique(
                dropna=False
            )
            == EXPECTED_ROWS,
        ),
        (
            "rcv_no_duplicates",
            duplicate_rcv == 0,
        ),
        (
            "rcv_no_malformed",
            malformed_rcv == 0,
        ),
        (
            "target_gene_resolved_all_rows",
            int(
                gene.eq("").sum()
            )
            == 0,
        ),
        (
            "gene_counts_exact",
            all(
                int(
                    observed_gene_counts.get(
                        key,
                        0,
                    )
                )
                == value
                for key, value
                in EXPECTED_GENE_COUNTS.items()
            ),
        ),
        (
            "axis_counts_exact",
            all(
                int(
                    observed_axis_counts.get(
                        key,
                        0,
                    )
                )
                == value
                for key, value
                in EXPECTED_AXIS_COUNTS.items()
            ),
        ),
        (
            "nested_scv_json_no_parse_errors",
            nested_parse_errors == 0,
        ),
        (
            "nested_scv_json_all_lists",
            nested_non_lists == 0,
        ),
        (
            "nested_scv_total_145400",
            nested_total
            == EXPECTED_NESTED_SCVS,
        ),
        (
            "nested_scv_counts_match_top_level",
            scv_count_mismatches == 0,
        ),
        (
            "scv_group_json_no_parse_errors",
            group_parse_errors == 0,
        ),
        (
            "scv_group_json_valid_dictionaries",
            group_invalid == 0,
        ),
        (
            "aggregate_conflict_positive_6602",
            conflict_positive
            == EXPECTED_CONFLICT_POSITIVE,
        ),
        (
            "empty_structured_condition_ids_730",
            empty_condition_ids
            == EXPECTED_EMPTY_CONDITION_IDS,
        ),
        (
            "review_stars_within_0_to_4",
            review_stars
            .dropna()
            .between(
                0,
                4,
                inclusive="both",
            )
            .all(),
        ),
        (
            "scv_count_nonnegative",
            scv_count
            .dropna()
            .ge(0)
            .all(),
        ),
        (
            "submitter_count_nonnegative",
            submitter_count
            .dropna()
            .ge(0)
            .all(),
        ),
        (
            "condition_source_resolved",
            resolved[
                "condition_names"
            ]
            is not None
            or resolved[
                "condition_ids"
            ]
            is not None,
        ),
        (
            "aggregate_interpretation_source_resolved",
            resolved[
                "aggregate_classification"
            ]
            is not None
            or resolved[
                "aggregate_classification_group"
            ]
            is not None,
        ),
        (
            "nested_scv_source_resolved",
            resolved[
                "nested_scvs"
            ]
            is not None,
        ),
        (
            "all_required_future_packet_fields_available",
            all_required_fields_available,
        ),
        (
            "field_inventory_matches_specification",
            len(field_derivability)
            == len(field_specs),
        ),
        (
            "four_score_fields_marked_prohibited",
            int(
                field_derivability[
                    "derivation_status"
                ]
                .eq(
                    "available_but_prohibited_in_7b1"
                )
                .sum()
            )
            == 4,
        ),
        (
            "sixteen_question_strata_defined",
            len(question_strata)
            == 16,
        ),
        (
            "five_questions_target_per_stratum",
            question_strata[
                "target_question_count"
            ]
            .eq(5)
            .all(),
        ),
        (
            "all_question_strata_feasible",
            all_strata_feasible,
        ),
        (
            "no_questions_selected_or_generated",
            question_strata[
                "questions_selected_or_generated"
            ]
            .eq(False)
            .all(),
        ),
        (
            "score_table_not_in_input_paths",
            PROHIBITED_CELL_7A3_SCORE_TABLE
            not in INPUT_PATHS.values(),
        ),
        (
            "score_columns_not_loaded",
            not any(
                column
                in t1.columns
                for column in [
                    "full_ges_p_stable_t1",
                    "full_ges_instability_risk_t1",
                    "no_star_ges_p_stable_t1",
                    "no_star_ges_instability_risk_t1",
                    "combined_metadata_instability_risk",
                ]
            ),
        ),
        (
            "no_row_level_output_defined",
            all(
                path.suffix.lower()
                != ".parquet"
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_evidence_packet_output_defined",
            not any(
                "packet"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_corpus_output_defined",
            not any(
                "corpus"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_embedding_output_defined",
            not any(
                "embedding"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_question_text_output_defined",
            not any(
                "question_set"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "no_prompt_output_defined",
            not any(
                "prompt"
                in path.name.lower()
                for path
                in OUTPUTS.values()
            ),
        ),
        (
            "egfr_remains_exploratory",
            json.loads(
                CELL_7B0_PROTOCOL.read_text(
                    encoding="utf-8"
                )
            )[
                "question_set"
            ][
                "egfr_role"
            ]
            == (
                "exploratory and "
                "separately reported"
            ),
        ),
    ]
)


failed_prewrite = [
    name
    for name, passed
    in prewrite_checks.items()
    if not bool(passed)
]


if failed_prewrite:
    raise RuntimeError(
        "Cell 7B1 failed before output. "
        "Failed checks:\n- "
        + "\n- ".join(
            failed_prewrite
        )
    )


# ============================================================
# FREEZE OUTPUTS
# ============================================================

stable_write_csv(
    OUTPUTS["source_inventory"],
    source_inventory,
)

stable_write_csv(
    OUTPUTS["field_derivability"],
    field_derivability,
)

stable_write_csv(
    OUTPUTS["gene_axis_inventory"],
    gene_axis_inventory,
)

stable_write_csv(
    OUTPUTS["question_strata"],
    question_strata,
)


preflight_report = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "upstream_cell_7b0_manifest_sha256": EXPECTED_HASHES[
        "cell_7b0_manifest"
    ],
    "raw_t1_parquet_sha256": EXPECTED_HASHES[
        "t1_parquet"
    ],
    "score_blind_boundary": {
        "cell_7a3_score_table_loaded": False,
        "ges_or_metadata_score_columns_loaded": False,
        "score_based_filtering_or_ranking": False,
    },
    "source_integrity": {
        "rows": len(t1),
        "columns": len(t1.columns),
        "unique_rcv_accessions": int(
            rcv.nunique(
                dropna=False
            )
        ),
        "malformed_rcv_accessions": malformed_rcv,
        "duplicate_rcv_rows": duplicate_rcv,
        "nested_scv_total": nested_total,
        "nested_scv_parse_errors": nested_parse_errors,
        "nested_scv_count_mismatches": scv_count_mismatches,
        "aggregate_conflict_positive": conflict_positive,
        "empty_structured_condition_ids": empty_condition_ids,
        "gene_counts": {
            key: int(
                observed_gene_counts.get(
                    key,
                    0,
                )
            )
            for key
            in EXPECTED_GENE_COUNTS
        },
        "classification_axis_counts": {
            key: int(
                observed_axis_counts.get(
                    key,
                    0,
                )
            )
            for key
            in EXPECTED_AXIS_COUNTS
        },
    },
    "field_derivability": {
        "audited_fields": int(
            len(field_derivability)
        ),
        "required_fields_available": all_required_fields_available,
        "score_dependent_fields_marked_prohibited": int(
            field_derivability[
                "derivation_status"
            ]
            .eq(
                "available_but_prohibited_in_7b1"
            )
            .sum()
        ),
    },
    "question_strata": {
        "strata": int(
            len(question_strata)
        ),
        "target_questions_per_stratum": 5,
        "all_strata_feasible": all_strata_feasible,
        "questions_selected_or_generated": False,
    },
    "scientific_operations": {
        "evidence_packets_materialized": False,
        "rag_corpus_constructed": False,
        "embeddings_generated": False,
        "retrieval_executed": False,
        "reranking_executed": False,
        "questions_selected_or_generated": False,
        "answer_key_constructed": False,
        "prompts_generated": False,
        "llm_called": False,
        "scores_loaded": False,
        "threshold_or_weight_optimized": False,
        "hard_exclusion_applied": False,
    },
}


stable_write_json(
    OUTPUTS["preflight_report"],
    preflight_report,
)


for key in [
    "source_inventory",
    "field_derivability",
    "gene_axis_inventory",
    "question_strata",
    "preflight_report",
]:
    write_sidecar(
        OUTPUTS[key]
    )


terminal_decision = (
    "PASS_STAGE7B1_SCORE_BLIND_RAW_T1_EVIDENCE_UNIT_FIELD_DERIVABILITY_"
    "ELIGIBILITY_AND_QUESTION_STRATUM_PREFLIGHT_FROZEN_CHECKSUM_PROTECTED_"
    "NO_CELL7A3_SCORES_LOADED_NO_EVIDENCE_PACKETS_CORPUS_EMBEDDINGS_"
    "RETRIEVAL_QUESTIONS_PROMPTS_OR_LLM_CELL7B2_SCORE_BLIND_FORMATTING_"
    "SAMPLING_ANSWER_KEY_AND_RUBRIC_PROTOCOL_FREEZE_ONLY_AUTHORIZED"
)


qc_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "passed_checks": len(
        prewrite_checks
    ),
    "failed_checks": [],
    "total_checks": len(
        prewrite_checks
    ),
    "checks": [
        {
            "check": key,
            "passed": bool(value),
        }
        for key, value
        in prewrite_checks.items()
    ],
    "decision": terminal_decision,
}


stable_write_json(
    OUTPUTS["qc"],
    qc_payload,
)

write_sidecar(
    OUTPUTS["qc"]
)


output_records = []


for key in [
    "source_inventory",
    "field_derivability",
    "gene_axis_inventory",
    "question_strata",
    "preflight_report",
    "qc",
]:
    path = OUTPUTS[key]

    if not sidecar_is_valid(path):
        raise AssertionError(
            "Output sidecar verification "
            f"failed for {key}: {path}"
        )

    output_records.append(
        {
            "artifact": key,
            "path": str(path),
            "sha256": sha256_file(
                path
            ),
            "sidecar_path": str(
                sidecar_path(path)
            ),
            "sidecar_sha256": sha256_file(
                sidecar_path(path)
            ),
        }
    )


next_authorized_cell = {
    "cell_id": "7B2",
    "scope": (
        "Score-blind evidence-packet formatting, "
        "deterministic eligibility and sampling, "
        "question-construction, answer-key, rubric, "
        "and adjudication protocol freeze only."
    ),
    "may_load_cell_7a3_scores": False,
    "may_materialize_evidence_packets": False,
    "may_construct_corpus": False,
    "may_generate_embeddings": False,
    "may_run_retrieval": False,
    "may_select_or_generate_questions": False,
    "may_construct_answer_key": False,
    "may_generate_prompts": False,
    "may_call_llm": False,
}


manifest_payload = {
    "cell_id": CELL_ID,
    "package_version": PACKAGE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "upstream_artifacts": [
        {
            "artifact": key,
            "path": str(path),
            "sha256": observed_hashes[
                key
            ],
        }
        for key, path
        in INPUT_PATHS.items()
    ],
    "output_artifacts": output_records,
    "qc": {
        "path": str(
            OUTPUTS["qc"]
        ),
        "sha256": sha256_file(
            OUTPUTS["qc"]
        ),
        "passed_checks": len(
            prewrite_checks
        ),
        "failed_checks": 0,
        "total_checks": len(
            prewrite_checks
        ),
    },
    "scientific_boundary": preflight_report[
        "scientific_operations"
    ],
    "terminal_decision": terminal_decision,
    "next_authorized_cell": next_authorized_cell,
}


stable_write_json(
    OUTPUTS["manifest"],
    manifest_payload,
)

write_sidecar(
    OUTPUTS["manifest"]
)


if not sidecar_is_valid(
    OUTPUTS["manifest"]
):
    raise AssertionError(
        "Cell 7B1 manifest sidecar "
        "verification failed."
    )


# ============================================================
# READBACK QC
# ============================================================

field_readback = pd.read_csv(
    OUTPUTS["field_derivability"]
)

strata_readback = pd.read_csv(
    OUTPUTS["question_strata"]
)

qc_readback = json.loads(
    OUTPUTS["qc"].read_text(
        encoding="utf-8"
    )
)

manifest_readback = json.loads(
    OUTPUTS["manifest"].read_text(
        encoding="utf-8"
    )
)


feasible_readback = (
    strata_readback[
        "feasible_for_target"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
    .all()
)


readback_checks = OrderedDict(
    [
        (
            "field_inventory_readback_matches_specification",
            len(field_readback)
            == len(field_specs),
        ),
        (
            "question_strata_readback_16_rows",
            len(strata_readback)
            == 16,
        ),
        (
            "question_strata_readback_all_feasible",
            feasible_readback,
        ),
        (
            "qc_readback_zero_failures",
            len(
                qc_readback.get(
                    "failed_checks",
                    [],
                )
            )
            == 0,
        ),
        (
            "manifest_readback_terminal_decision",
            manifest_readback.get(
                "terminal_decision"
            )
            == terminal_decision,
        ),
        (
            "manifest_readback_authorizes_7b2",
            manifest_readback.get(
                "next_authorized_cell",
                {},
            ).get(
                "cell_id"
            )
            == "7B2",
        ),
        (
            "manifest_readback_7b2_score_blind",
            manifest_readback.get(
                "next_authorized_cell",
                {},
            ).get(
                "may_load_cell_7a3_scores"
            )
            is False,
        ),
        (
            "manifest_sidecar_valid",
            sidecar_is_valid(
                OUTPUTS["manifest"]
            ),
        ),
    ]
)


failed_readback = [
    name
    for name, passed
    in readback_checks.items()
    if not bool(passed)
]


if failed_readback:
    raise RuntimeError(
        "Cell 7B1 failed during readback "
        "verification. Failed checks:\n- "
        + "\n- ".join(
            failed_readback
        )
    )


immutable_hashes_after = {
    key: sha256_file(path)
    for key, path
    in INPUT_PATHS.items()
}


if (
    immutable_hashes_after
    != immutable_hashes_before
):
    raise AssertionError(
        "A frozen upstream artifact "
        "changed during Cell 7B1."
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

total_checks = (
    len(prewrite_checks)
    + len(readback_checks)
)


separator = "=" * 144


print(
    "\n"
    + separator
)

print(
    "EXPERIMENT 2 — STAGE 7B — CELL 7B1"
)

print(
    "SCORE-BLIND EVIDENCE-UNIT, "
    "FIELD-DERIVABILITY, ELIGIBILITY, "
    "AND QUESTION-STRATUM PREFLIGHT"
)

print(separator)

print(
    f"Notebook                                      : "
    f"{NOTEBOOK_NAME}"
)

print(
    f"Project root                                  : "
    f"{ROOT}"
)


print(
    "\nUPSTREAM AUTHORIZATION"
)

print(
    f"Cell 7B0 manifest SHA-256                     : "
    f"{sha256_file(CELL_7B0_MANIFEST)}"
)

print(
    "Cell 7B0 terminal PASS verified               : YES"
)

print(
    f"Cell 7B0 manifest QC                          : "
    f"{manifest_qc_passed}/"
    f"{manifest_qc_total} PASS"
)

print(
    f"Cell 7B0 QC record                            : "
    f"{record_qc_passed}/"
    f"{record_qc_total} PASS"
)

print(
    "Cell 7A3 score loading                        : NO"
)


print(
    "\nRAW T1 SOURCE REVERIFICATION"
)

print(
    f"T1 Parquet SHA-256                            : "
    f"{sha256_file(T1_PARQUET)}"
)

print(
    f"Rows                                           : "
    f"{len(t1):,}"
)

print(
    f"Columns                                        : "
    f"{len(t1.columns)}"
)

print(
    f"Unique RCV accessions                         : "
    f"{rcv.nunique(dropna=False):,}"
)

print(
    f"Malformed RCV accessions                      : "
    f"{malformed_rcv:,}"
)

print(
    f"Duplicate RCV rows                            : "
    f"{duplicate_rcv:,}"
)

print(
    f"Nested SCVs                                   : "
    f"{nested_total:,}"
)

print(
    f"Nested-SCV parse errors                       : "
    f"{nested_parse_errors:,}"
)

print(
    f"Nested-SCV count mismatches                   : "
    f"{scv_count_mismatches:,}"
)

print(
    f"Aggregate conflict-positive RCVs              : "
    f"{conflict_positive:,}"
)

print(
    f"Empty structured condition-ID lists           : "
    f"{empty_condition_ids:,}"
)


print(
    "\nFIELD DERIVABILITY"
)

print(
    f"Fields audited                                : "
    f"{len(field_derivability)}"
)

print(
    f"Required future packet fields available       : "
    f"{'YES' if all_required_fields_available else 'NO'}"
)

print(
    "Cell 7A3 score-dependent fields loaded        : NO"
)

print(
    "Row-level evidence packet saved               : NO"
)


print(
    "\nQUESTION-STRATUM PREFLIGHT"
)


for row in question_strata.to_dict(
    "records"
):
    label = (
        f"{row['target_gene']} | "
        f"{row['question_type']}"
    )

    feasibility = (
        "YES"
        if row["feasible_for_target"]
        else "NO"
    )

    print(
        f"{label:<46}: "
        f"{int(row['eligible_rcv_count']):,} eligible | "
        f"target=5 | "
        f"feasible={feasibility}"
    )


print(
    "Questions selected or generated               : NO"
)


print(
    "\nCELL 7B1 FROZEN OUTPUTS"
)


for label, key in [
    (
        "Score-blind source inventory",
        "source_inventory",
    ),
    (
        "Evidence-field derivability inventory",
        "field_derivability",
    ),
    (
        "Gene-axis eligibility inventory",
        "gene_axis_inventory",
    ),
    (
        "Question-stratum availability inventory",
        "question_strata",
    ),
    (
        "Preflight report",
        "preflight_report",
    ),
    (
        "QC record",
        "qc",
    ),
    (
        "Manifest",
        "manifest",
    ),
]:
    path = OUTPUTS[key]

    print(
        f"{label:<46}: {path}"
    )

    print(
        f"{'SHA-256':<46}: "
        f"{sha256_file(path)}"
    )


print(
    f"\nQC checks                                      : "
    f"{total_checks}/{total_checks} PASS"
)


print(
    "\nSCIENTIFIC OPERATIONS"
)

print(
    "Cell 7A3 scores loaded                        : NO"
)

print(
    "Evidence packets materialized                 : NO"
)

print(
    "RAG corpus constructed                        : NO"
)

print(
    "Embeddings generated                          : NO"
)

print(
    "Retrieval or reranking executed               : NO"
)

print(
    "Questions selected or generated               : NO"
)

print(
    "Answer key or rubric constructed              : NO"
)

print(
    "Prompts generated                             : NO"
)

print(
    "LLM called                                     : NO"
)

print(
    "Threshold or weight optimization              : NO"
)

print(
    "Hard evidence exclusion applied               : NO"
)


print(
    "\nNEXT AUTHORIZED CELL"
)

print(
    "Cell 7B2                                      : "
    "Score-blind formatting, sampling,"
)

print(
    "                                                 "
    "question-construction, answer-key,"
)

print(
    "                                                 "
    "rubric, and adjudication protocol freeze only"
)

print(
    "Cell 7A3 score loading                        : PROHIBITED"
)

print(
    "Evidence-packet materialization               : PROHIBITED"
)

print(
    "Corpus / embeddings / retrieval               : PROHIBITED"
)

print(
    "Question selection or generation              : PROHIBITED"
)

print(
    "Prompt / LLM generation                       : PROHIBITED"
)


print(
    f"\nFINAL DECISION                                : "
    f"{terminal_decision}"
)

print(separator)


EXPERIMENT 2 — STAGE 7B — CELL 7B1
SCORE-BLIND EVIDENCE-UNIT, FIELD-DERIVABILITY, ELIGIBILITY, AND QUESTION-STRATUM PREFLIGHT
Notebook                                      : 05_GES_Aware_Genomic_RAG_Cell_7B1_V3.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM AUTHORIZATION
Cell 7B0 manifest SHA-256                     : df9342b8a2fb641f4cb68ff18ae9568bb7f63eff3fcc5e1ae1106601fa59e42a
Cell 7B0 terminal PASS verified               : YES
Cell 7B0 manifest QC                          : 50/50 PASS
Cell 7B0 QC record                            : 50/50 PASS
Cell 7A3 score loading                        : NO

RAW T1 SOURCE REVERIFICATION
T1 Parquet SHA-256                            : 5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
Rows                                           : 100,920
Columns                                        : 36
Unique RCV accessions                         : 100,920
Malformed RCV accessio